# Week 08 · RAG：证据、引用与评测

RAG 分为检索与生成。解析阶段保留原文页码和段落，chunking 影响可找到的证据，top-k 和上下文预算决定交给生成器的内容。来源 ID 应稳定，不能用页面展示顺序冒充原始来源。上传去重依赖内容哈希；相同文件名不一定相同内容。

本例是可离线验证的 extractive QA 基线：返回来源原句，证据不足拒答，不伪装成 LLM 生成。网站 Codex 助手则可接入真正的生成步骤。文档中的“忽略规则”是数据，不是指令。引用存在只证明编号有效，不证明该片段真的蕴含答案；需要独立评测。

检索命中率衡量正确片段能否进入 top-k，答案正确率衡量内容，引用支持率衡量论据对应。至少 30 道评测题应涵盖可回答、同义改写、跨文档、冲突和无答案。本例的模板生成题用于回归测试，不应冒充人工标注的真实评测集。

## 学习方式 / How to study
先预测代码结果，再逐行运行。改变一个输入、解释变化，最后不看参考实现重写关键函数。阅读不是掌握的证据；能独立实现、测试、解释失败才是。

In [ ]:
import hashlib,json,re
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

pages=[("python.md",1,"Python virtual environments isolate dependencies for each project."),
       ("sql.md",1,"A database transaction groups changes so they commit or roll back together."),
       ("http.md",1,"HTTP 404 means the requested resource was not found."),
       ("rag.md",2,"RAG retrieves source passages before generating a grounded answer."),
       ("testing.md",3,"Unit tests isolate business logic; integration tests check component boundaries.")]
sources={}
for filename,page,content in pages:
    digest=hashlib.sha256((filename+str(page)+content).encode()).hexdigest()[:12]
    sources[digest]={"file":filename,"page":page,"paragraph":1,"content":content}
ids=list(sources)
vectorizer=TfidfVectorizer(stop_words="english")
matrix=vectorizer.fit_transform([sources[i]["content"] for i in ids])
history=[]
def answer(question):
    vector=vectorizer.transform([question])
    scores=cosine_similarity(vector,matrix).ravel()
    best=int(scores.argmax())
    if scores[best]<.12:
        result={"answer":"Insufficient evidence.","source":None}
    else:
        identifier=ids[best]
        result={"answer":sources[identifier]["content"],"source":identifier,**sources[identifier]}
    history.append({"question":question,**result})
    return result
print(answer("What does HTTP 404 mean?"))
assert answer("Who won the football championship?")["source"] is None
# 5 个知识点 × 6 种模板 = 30 个回归问题，显式标为 synthetic。
topics=["Python virtual environments","database transaction","HTTP 404","RAG source passages","Unit tests"]
templates=["Explain {}", "What is {}?", "Describe {}", "Give evidence for {}", "Summarize {}", "Define {}"]
evaluation=[]
for i,topic in enumerate(topics):
    for template in templates:
        question=template.format(topic);result=answer(question)
        evaluation.append({"question":question,"expected_source":ids[i],"actual_source":result["source"],"correct":result["source"]==ids[i],"synthetic":True})
report=pd.DataFrame(evaluation)
assert len(report)==30
display(report)
print("检索命中率：",report.correct.mean())
report.to_csv("week08-evaluation.csv",index=False)
Path("week08-history.json").write_text(json.dumps(history,indent=2),encoding="utf-8")

## 练习 / Exercises
手工写 10 道模板没有覆盖的问题，至少两道无法回答。不要调低阈值来强行提高可回答数量。

先在下面独立完成，再展开参考实现。

In [ ]:
# 在这里写你的实现；运行后检查边界。


## 参考实现与验收 / Reference and checks
参考实现是一个可行方案，不是唯一答案。不要在未完成练习前直接复制。

In [ ]:
for question in ["What happens when a transaction fails?","How do I train a large language model?","What does missing resource mean?"]:
    result=answer(question)
    print(question,result)
    if result["source"] is not None:
        assert result["source"] in sources
        assert result["answer"]==sources[result["source"]]["content"]
print("来源存在检查通过；语义支持仍需人工核对。")